<br/>

<div align="center">
<span style="font-size: 2.5em;">Model Performance Comparison</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Comprehensive evaluation of trained neural network posteriors across data regimes</span>
</div>

In [1]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os, sys
desired_root_name = "dark-matter-sbi"  
while os.path.basename(os.getcwd()) != desired_root_name:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

import torch
from tabulate import tabulate
from configs.config import load_model, MODEL_CONFIG
from performances.performances import get_or_evaluate_performances, print_performance_tables

## Overview

This notebook compares SBI model performance across architectures and data regimes.

**Structure:**

1. **Training Metrics**: Validation loss and accuracy from checkpoints
2. **Posterior Quality Metrics**: Coverage, JSD, and Wasserstein distances
3. **Distance Metrics**: Euclidean and Mahalanobis distances (mean/median)

Models are evaluated across three data regimes (low, mid, high) that represent different parameter ranges and event statistics.

## Training Metrics: Loss and Accuracy

Compare validation loss and accuracy across architectures and data regimes. These metrics provide a quick overview of training convergence and basic classification performance.

In [2]:
# ===========================================
# CONFIGURATION
# ===========================================
        
halo      = "default"       
datatags = ["low", "mid", "high"]
n_train   = 300_000
top_k     = 10              

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ===========================================
# LOAD MODELS AND EXTRACT METRICS
# ===========================================

results = []

for modelname in MODEL_CONFIG.keys():
    for datatag in datatags:

        MODELPATH = f"models/wimpy/{halo}/{modelname}_n{n_train}_{datatag}_{halo}.pt"
        if not os.path.exists(MODELPATH):
            continue

        # Load model checkpoint
        model, ckpt = load_model(MODELPATH, modelname, print_arch=False)

        # Extract best validation metrics
        best_val_loss = ckpt.get("best_val_loss")
        best_val_acc  = ckpt.get("best_val_acc")

        results.append({
            "model": modelname,
            "datatag": datatag,
            "best_val_loss": best_val_loss,
            "best_val_acc": best_val_acc,
        })


# ===========================================
# DISPLAY FUNCTION
# ===========================================

def print_side_by_side_tables(results, datatags):
    """
    Print performance tables for each data regime side-by-side.
    
    Models are sorted by accuracy (descending) then loss (ascending).
    """
    tables = []
    print(f"\nn_train={n_train}\n")
    
    for tag in datatags:
        # Filter and sort results for this datatag
        rows = [
            [r["model"], r["best_val_loss"], r["best_val_acc"]]
            for r in results
            if r["datatag"] == tag
        ]

        rows_sorted = sorted(
            rows,
            key=lambda x: (-(x[2] or -1), x[1] or float("inf"))
        )

        # Create table
        table = tabulate(
            rows_sorted,
            headers=[f"{tag.upper()} Model", "Best Val Loss", "Best Val Acc"],
            floatfmt=".3f",
            tablefmt="grid"
        )
        tables.append(table.splitlines())

    # Align tables to same height
    max_height = max(len(t) for t in tables)
    for t in tables:
        while len(t) < max_height:
            t.append(" " * len(t[0]))

    # Print tables side-by-side
    for lines in zip(*tables):
        print("          ".join(lines))

In [3]:
print_side_by_side_tables(results, datatags)


n_train=300000

+-------------+-----------------+----------------+          +-------------+-----------------+----------------+          +--------------+-----------------+----------------+
| LOW Model   |   Best Val Loss |   Best Val Acc |          | MID Model   |   Best Val Loss |   Best Val Acc |          | HIGH Model   |   Best Val Loss |   Best Val Acc |
+=============+=================+================+          +=============+=================+================+          +==============+=================+================+
| ntothighest |           0.376 |          0.801 |          | ntothighest |           0.234 |          0.893 |          | ntothighest  |           0.116 |          0.956 |
+-------------+-----------------+----------------+          +-------------+-----------------+----------------+          +--------------+-----------------+----------------+
| vanilla     |           0.379 |          0.800 |          | full        |           0.235 |          0.893 |          | f

## Advanced Performance Metrics

Load pre-calculated evaluation results including coverage tests, distance metrics, and uncertainty quantification. These metrics assess posterior quality beyond simple classification accuracy.

In [6]:
# ===========================================
# CONFIGURATION
# ===========================================

datatags = ["low", "mid", "high"]
n_train  = 300_000
n_total  = 20000
split    = "val"


# ===========================================
# LOAD PRE-CALCULATED RESULTS
# ===========================================

results, meta = get_or_evaluate_performances(
    datatags=datatags,
    n_train=n_train,
    n_total=n_total,
    split=split,
    specific_model=None,  # Evaluate all models
)


# ===========================================
# DISPLAY RESULTS
# ===========================================

print_performance_tables(results, meta, datatags)

[INFO] Loading cached results: performances/performances_results\perf_default_all_n300000_N20000_val.pt

Meta: {'n_train': 300000, 'n_total': 20000, 'split': 'val', 'halo': 'default', 'specific_model': None}

=== LOW ===
╒═════════════╤═══════════╤══════════╤══════════╤═════════════╤═══════╤═══════╤════════╤═════════╤══════════╤═══════════╤══════════╤═══════════╕
│ Model       │   ValLoss │   ValAcc │   CvgAbs │   CvgSigned │   JSD │   SWD │   W(m) │   W(cp) │   EucMed │   EucMean │   MahMed │   MahMean │
╞═════════════╪═══════════╪══════════╪══════════╪═════════════╪═══════╪═══════╪════════╪═════════╪══════════╪═══════════╪══════════╪═══════════╡
│ ntothighest │     0.376 │    0.801 │    0.006 │       0.006 │ 0.013 │ 0.409 │  0.468 │   0.372 │    0.222 │     0.393 │    1.275 │     1.430 │
├─────────────┼───────────┼──────────┼──────────┼─────────────┼───────┼───────┼────────┼─────────┼──────────┼───────────┼──────────┼───────────┤
│ vanilla     │     0.379 │    0.800 │    0.011 │     